In [1]:
import os
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType, DoubleType, DateType, TimestampType
from pyspark.sql.functions import col



import pyspark.sql.functions as F
from pyspark.sql import Window
import json

In [2]:
sc_conf = SparkConf()\
    .set("spark.executor.memory", "4g")\
    .set("spark.executor.cores", "2")\
    .set("spark.hadoop.io.nativeio.enabled", "false")
#add extra parameters if required

In [3]:
spark = SparkSession.builder\
	.appName("Customer Name Cleanup")\
	.config(conf=sc_conf)\
	.getOrCreate()

In [4]:
data = [
    (1, "John Doe", 1000),
    (2, "Jane#Smith", 2500),
    (3, "Bob@Johnson", 500)
]

schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("balance", IntegerType(), True)
])

NameError: name 'df' is not defined

In [ ]:
df = spark.createDataFrame(data, schema)
# find and replace non-alphabetic characters in the customer_name column with a space
cleaned_df = df.withColumn("customer_name", F.regexp_replace(col("customer_name"), r"[^a-zA-Z\s]", " "))
cleaned_df.show(truncate=False)

+-----------+-------------+-------+
|customer_id|customer_name|balance|
+-----------+-------------+-------+
|1          |John Doe     |1000   |
|2          |Jane Smith   |2500   |
|3          |Bob Johnson  |500    |
+-----------+-------------+-------+



In [7]:
# how to find the names with special characters in the customer_name column ?
# To find the names with special characters in the `customer_name` column, you can use the `regexp_extract` function from PySpark. This function allows you to extract substrings that match a specified regular expression pattern. In this case, you can use a regular expression to identify names that contain special characters.

from pyspark.sql.functions import regexp_extract
# Define a regular expression pattern to match names with special characters
pattern = r'[^a-zA-Z\s]'

# Use regexp_extract to find names with special characters
special_char_names_df = df.withColumn("has_special_char", regexp_extract(col("customer_name"), pattern, 0) != "")

# Filter the DataFrame to get only the names with special characters
special_char_names_df = special_char_names_df.filter(col("has_special_char") == True)
# Show the names with special characters
special_char_names_df.show(truncate=False)

+-----------+-------------+-------+----------------+
|customer_id|customer_name|balance|has_special_char|
+-----------+-------------+-------+----------------+
|2          |Jane#Smith   |2500   |true            |
|3          |Bob@Johnson  |500    |true            |
+-----------+-------------+-------+----------------+



In [ ]:
df.exceptAll(cleaned_df).show(truncate=False)

+-----------+-------------+-------+
|customer_id|customer_name|balance|
+-----------+-------------+-------+
|2          |Jane#Smith   |2500   |
|3          |Bob@Johnson  |500    |
+-----------+-------------+-------+



----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 62335)
Traceback (most recent call last):
  File "c:\Users\Jabir\AppData\Local\Programs\Python\Python310\lib\socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "c:\Users\Jabir\AppData\Local\Programs\Python\Python310\lib\socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "c:\Users\Jabir\AppData\Local\Programs\Python\Python310\lib\socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "c:\Users\Jabir\AppData\Local\Programs\Python\Python310\lib\socketserver.py", line 747, in __init__
    self.handle()
  File "c:\Users\Jabir\AppData\Local\Programs\Python\Python310\lib\site-packages\pyspark\accumulators.py", line 281, in handle
    poll(accum_updates)
  File "c:\Users\Jabir\AppData\Local\Programs\Python\Python310\lib